### Evaluation

### Evaluation Metric Properties

#### Formulate

**Mean Absolute Error (MAE)**

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**Mean Squared Error (MSE)**

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

where $y_i$ is the true value, $\hat{y}_i$ is the predicted value, and $n$
is the number of instances.

#### Why would someone use one and not the other?

The key difference lies in how each metric treats errors of different
magnitudes. MAE weights all errors linearly — an error of 2.0 is exactly
twice as costly as an error of 1.0. MSE weights errors quadratically — an
error of 2.0 is four times as costly as an error of 1.0.

This has three practical consequences:

**1. Sensitivity to outliers.**
MSE disproportionately penalises large errors. A single prediction that is
scenarios contribute equally (3 in total). If the application cannot
tolerate large individual errors (e.g., safety-critical systems), MSE is
preferred because it drives the optimiser to avoid worst-case predictions.
If typical performance matters more than worst-case performance (e.g.,
mood tracking where occasional outliers are acceptable), MAE is preferred.

**2. Robustness to noise.**
In datasets with heavy-tailed error distributions or noisy targets, MSE
can be dominated by a few extreme residuals that may reflect measurement
noise rather than systematic model failure. MAE is more robust in such
settings because it does not amplify these extreme values.

**3. Optimisation behaviour.**
A model optimised for MSE (e.g., ordinary least squares) finds the
conditional mean of the target distribution. A model optimised for MAE
finds the conditional median. When the target distribution is symmetric,
these coincide. When the distribution is skewed, they diverge — and the
choice of metric implicitly selects which central tendency the model
targets.

#### Example: when MAE and MSE give identical results

MAE and MSE produce identical rankings (and identical optimal predictions)
when **all prediction errors have exactly the same absolute magnitude**.

Consider a dataset with $n$ instances where a model produces errors
$e_i = y_i - \hat{y}_i$. If $|e_i| = c$ for all $i$ (i.e., every
error has the same absolute size $c$, though signs may differ), then:

$$\text{MAE} = \frac{1}{n} \sum |e_i| = \frac{1}{n} \cdot n \cdot c = c$$

$$\text{MSE} = \frac{1}{n} \sum e_i^2 = \frac{1}{n} \cdot n \cdot c^2 = c^2$$

Since MSE $= c^2 = \text{MAE}^2$, both metrics are monotonic
transformations of each other. Any model that minimises one also minimises
the other. In particular, if we compare two models A and B:

- If all errors of A have magnitude $c_A$ and all errors of B have
  magnitude $c_B$, then $\text{MAE}_A < \text{MAE}_B$ if and only if
  $\text{MSE}_A < \text{MSE}_B$.

A concrete instance: a binary classification problem where the target is 0
or 1, the model always predicts 0.5, and every instance is either 0 or 1.
Every error is exactly $|0.5| = 0.5$. MAE $= 0.5$, MSE $= 0.25 = 0.5^2$.
No alternative constant prediction can improve one metric without also
improving the other.

More generally, whenever the error distribution has **zero variance**
(all residuals are the same size), MAE and MSE are equivalent for model
comparison.

We now apply MAE and MSE to the regression results from the regression step and
analyse how the choice of metric affects model evaluation and
interpretation.

#### 1. Load Regression Results

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = Path("../outputs")
FIG_DIR    = Path("../figures")

# Load per-instance errors from Task 4
err_xgb  = pd.read_csv(OUTPUT_DIR / "regression_errors_xgb.csv")
err_lstm = pd.read_csv(OUTPUT_DIR / "regression_errors_lstm.csv")

print(f"XGBoost: {len(err_xgb)} test predictions")
print(f"LSTM:    {len(err_lstm)} test predictions")

XGBoost: 1191 test predictions
LSTM:    530 test predictions


#### 2. MAE vs MSE — Overall Comparison

In [2]:
results = {}
for name, err_col, df_err in [("XGBoost", "error_xgb", err_xgb),
                                ("LSTM", "error_lstm", err_lstm)]:
    errors = df_err[err_col].values
    abs_err = np.abs(errors)
    sq_err  = errors ** 2

    mae = abs_err.mean()
    mse = sq_err.mean()
    rmse = np.sqrt(mse)
    median_ae = np.median(abs_err)

    results[name] = {
        "MAE": mae, "MSE": mse, "RMSE": rmse,
        "Median AE": median_ae, "errors": errors,
        "abs_err": abs_err, "sq_err": sq_err,
    }

print(f"{'Metric':<15s}  {'XGBoost':>10s}  {'LSTM':>10s}  {'Difference':>12s}")
print(f"{'─'*50}")
for metric in ["MAE", "MSE", "RMSE", "Median AE"]:
    xv = results["XGBoost"][metric]
    lv = results["LSTM"][metric]
    diff = xv - lv
    winner = "XGB" if diff < 0 else "LSTM"
    print(f"{metric:<15s}  {xv:>10.4f}  {lv:>10.4f}  {diff:>+10.4f} ({winner})")

Metric              XGBoost        LSTM    Difference
──────────────────────────────────────────────────
MAE                  0.4823      0.4601     +0.0222 (LSTM)
MSE                  0.4360      0.3938     +0.0422 (LSTM)
RMSE                 0.6603      0.6275     +0.0327 (LSTM)
Median AE            0.3699      0.3494     +0.0206 (LSTM)


#### 3. How MSE Amplifies Large Errors

The central question of Metric Sensitivity Analysis is: **does the choice of MAE vs MSE change
which model appears better, or how we interpret model performance?**

To answer this, we examine how each metric distributes its "penalty budget"
across errors of different sizes.

In [3]:
print(f"{'='*65}")
print("Error distribution analysis")
print(f"{'='*65}")

for name in ["XGBoost", "LSTM"]:
    r = results[name]
    abs_err = r["abs_err"]
    sq_err  = r["sq_err"]
    n = len(abs_err)

    # Contribution of top-5% errors
    p95 = np.percentile(abs_err, 95)
    top5 = abs_err >= p95
    n_top5 = top5.sum()

    mae_share = abs_err[top5].sum() / abs_err.sum() * 100
    mse_share = sq_err[top5].sum() / sq_err.sum() * 100

    # Contribution of bottom-50% errors
    p50 = np.median(abs_err)
    bot50 = abs_err <= p50
    mae_bot50 = abs_err[bot50].sum() / abs_err.sum() * 100
    mse_bot50 = sq_err[bot50].sum() / sq_err.sum() * 100

    print(f"\n{name}:")
    print(f"  Total instances:        {n}")
    print(f"  Mean |error|:           {abs_err.mean():.4f}")
    print(f"  Median |error|:         {np.median(abs_err):.4f}")
    print(f"  Max |error|:            {abs_err.max():.4f}")
    print(f"  Std of |error|:         {abs_err.std():.4f}")
    print(f"  95th percentile:        {p95:.4f}")
    print()
    print(f"  Top 5% of errors ({n_top5} instances):")
    print(f"    Share of MAE:         {mae_share:5.1f}%  "
          f"(proportional = 5%,  {mae_share/5:.1f}× overweight)")
    print(f"    Share of MSE:         {mse_share:5.1f}%  "
          f"(proportional = 5%,  {mse_share/5:.1f}× overweight)")
    print(f"    MSE/MAE ratio:        {mse_share/mae_share:.2f}×")
    print()
    print(f"  Bottom 50% of errors:")
    print(f"    Share of MAE:         {mae_bot50:5.1f}%")
    print(f"    Share of MSE:         {mse_bot50:5.1f}%")

Error distribution analysis

XGBoost:
  Total instances:        1191
  Mean |error|:           0.4823
  Median |error|:         0.3699
  Max |error|:            3.4948
  Std of |error|:         0.4510
  95th percentile:        1.2467

  Top 5% of errors (60 instances):
    Share of MAE:          18.6%  (proportional = 5%,  3.7× overweight)
    Share of MSE:          40.5%  (proportional = 5%,  8.1× overweight)
    MSE/MAE ratio:        2.17×

  Bottom 50% of errors:
    Share of MAE:          16.8%
    Share of MSE:           4.4%

LSTM:
  Total instances:        530
  Mean |error|:           0.4601
  Median |error|:         0.3494
  Max |error|:            3.4107
  Std of |error|:         0.4268
  95th percentile:        1.2221

  Top 5% of errors (27 instances):
    Share of MAE:          19.0%  (proportional = 5%,  3.8× overweight)
    Share of MSE:          42.2%  (proportional = 5%,  8.4× overweight)
    MSE/MAE ratio:        2.23×

  Bottom 50% of errors:
    Share of MAE:       

#### 4. Visualisation - Cumulative Error Contribution

This plot shows how each metric accumulates its total value across errors
sorted from smallest to largest. A steeper rise at the right end means
the metric is more sensitive to large errors.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, name in zip(axes, ["XGBoost", "LSTM"]):
    abs_err = np.sort(results[name]["abs_err"])
    sq_err  = np.sort(results[name]["abs_err"] ** 2)  # sorted by abs magnitude
    n = len(abs_err)

    # Cumulative shares
    cum_mae = np.cumsum(abs_err) / abs_err.sum() * 100
    cum_mse = np.cumsum(sq_err) / sq_err.sum() * 100
    pct = np.arange(1, n + 1) / n * 100

    ax.plot(pct, cum_mae, label="MAE (cumulative)", linewidth=2)
    ax.plot(pct, cum_mse, label="MSE (cumulative)", linewidth=2)
    ax.plot([0, 100], [0, 100], "k--", alpha=0.3, label="Proportional")
    ax.axvline(95, color="red", alpha=0.3, linestyle=":", label="95th percentile")

    ax.set_xlabel("Errors sorted by magnitude (percentile)")
    ax.set_ylabel("Cumulative share of total metric (%)")
    ax.set_title(f"{name}")
    ax.legend(fontsize=9)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)

fig.suptitle("How MAE and MSE distribute penalty across error magnitudes",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "cumulative_error_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure to {FIG_DIR / 'cumulative_error_comparison.png'}")

Saved figure to ..\figures\task5b_cumulative_error.png


C:\Users\babes\AppData\Local\Temp\ipykernel_38016\1096631075.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


#### 5. Does the Metric Choice Change Model Ranking?

In [5]:
print(f"\n{'='*65}")
print("Model ranking under different metrics")
print(f"{'='*65}")

# Mean-of-folds from Task 4 (loaded from regression_results.csv for precision)
reg_results = pd.read_csv(OUTPUT_DIR / "regression_results.csv")

print(f"\n  Mean-of-folds (from Task 4 walk-forward CV):")
for _, row in reg_results.iterrows():
    print(f"    {row['model']:<25s}  MAE={row['mae_mean']:.4f}  MSE={row['mse_mean']:.4f}")

# Pooled metrics
xgb_pooled_mae = np.abs(err_xgb["error_xgb"]).mean()
xgb_pooled_mse = (err_xgb["error_xgb"] ** 2).mean()
lstm_pooled_mae = np.abs(err_lstm["error_lstm"]).mean()
lstm_pooled_mse = (err_lstm["error_lstm"] ** 2).mean()

print(f"\n  Pooled metrics:")
print(f"    {'XGBoost':<25s}  MAE={xgb_pooled_mae:.4f}  MSE={xgb_pooled_mse:.4f}")
print(f"    {'LSTM':<25s}  MAE={lstm_pooled_mae:.4f}  MSE={lstm_pooled_mse:.4f}")

# Ranking
mae_winner = "LSTM" if lstm_pooled_mae < xgb_pooled_mae else "XGBoost"
mse_winner = "LSTM" if lstm_pooled_mse < xgb_pooled_mse else "XGBoost"

print(f"\n  Best model by pooled MAE: {mae_winner} "
      f"(Δ = {abs(xgb_pooled_mae - lstm_pooled_mae):.4f})")
print(f"  Best model by pooled MSE: {mse_winner} "
      f"(Δ = {abs(xgb_pooled_mse - lstm_pooled_mse):.4f})")

if mae_winner == mse_winner:
    print(f"\n  → MAE and MSE agree on the best model ({mae_winner}).")
else:
    print(f"\n  → MAE and MSE DISAGREE on the best model!")


Model ranking under different metrics

  Mean-of-folds (from Task 4 walk-forward CV):
    naive_baseline             MAE=0.5468  MSE=0.5552
    rolling_w5_baseline        MAE=0.4979  MSE=0.4491
    xgboost                    MAE=0.4877  MSE=0.4475
    lstm                       MAE=0.4961  MSE=0.4349

  Pooled metrics:
    XGBoost                    MAE=0.4823  MSE=0.4360
    LSTM                       MAE=0.4601  MSE=0.3938

  Best model by pooled MAE: LSTM (Δ = 0.0222)
  Best model by pooled MSE: LSTM (Δ = 0.0422)

  → MAE and MSE agree on the best model (LSTM).


#### 6. Per-Patient Error Analysis — Where MSE and MAE Diverge Most

In [6]:
# For XGBoost: compare per-patient MAE vs per-patient MSE
# to show that MSE amplifies patient-level heterogeneity

# We need patient IDs — load original modelling dataset
df_model = pd.read_csv(OUTPUT_DIR / "data_modelling.csv",
                        parse_dates=["date", "target_date"])
df_model = df_model.sort_values(["date", "id"]).reset_index(drop=True)

# The error file has the same row order as the test instances
# Reconstruct by matching — errors are from walk-forward folds 1-4 test sets
unique_dates = sorted(df_model["date"].unique())
n_dates = len(unique_dates)
block_size = n_dates // 5

date_blocks = []
for i in range(5):
    start = i * block_size
    end = (i + 1) * block_size if i < 4 else n_dates
    date_blocks.append(set(unique_dates[start:end]))

test_indices = []
for k in range(4):
    test_dates = date_blocks[k + 1]
    fold_idx = df_model[df_model["date"].isin(test_dates)].index.tolist()
    test_indices.extend(fold_idx)

# Attach patient IDs to errors
err_xgb_with_id = err_xgb.copy()
err_xgb_with_id["id"] = df_model.loc[test_indices, "id"].values

per_patient = err_xgb_with_id.groupby("id").agg(
    n=("error_xgb", "count"),
    patient_mae=("error_xgb", lambda x: np.abs(x).mean()),
    patient_mse=("error_xgb", lambda x: (x ** 2).mean()),
).reset_index()

per_patient["mse_mae_ratio"] = per_patient["patient_mse"] / per_patient["patient_mae"]
per_patient = per_patient.sort_values("mse_mae_ratio", ascending=False)

print(f"Per-patient MSE/MAE ratio — XGBoost:")
print(f"{'Patient':<12s}  {'N':>4s}  {'MAE':>7s}  {'MSE':>7s}  {'MSE/MAE':>8s}")
print(f"{'─'*45}")
for _, row in per_patient.iterrows():
    print(f"{row['id']:<12s}  {int(row['n']):>4d}  "
          f"{row['patient_mae']:>7.4f}  {row['patient_mse']:>7.4f}  "
          f"{row['mse_mae_ratio']:>8.3f}")

print(f"\nRange of MSE/MAE ratio: "
      f"{per_patient['mse_mae_ratio'].min():.3f} – "
      f"{per_patient['mse_mae_ratio'].max():.3f}")
print(f"Patients with MSE/MAE > 1.0 (large errors amplified): "
      f"{(per_patient['mse_mae_ratio'] > 1.0).sum()} / "
      f"{len(per_patient)}")

Per-patient MSE/MAE ratio — XGBoost:
Patient          N      MAE      MSE   MSE/MAE
─────────────────────────────────────────────
AS14.07         45   0.9696   1.6120     1.663
AS14.05         45   0.5223   0.8007     1.533
AS14.33         42   0.6883   0.8309     1.207
AS14.13         44   0.6524   0.7736     1.186
AS14.02         35   0.5813   0.6229     1.072
AS14.12         40   0.5563   0.5210     0.937
AS14.32         38   0.6295   0.5887     0.935
AS14.09         45   0.5797   0.5293     0.913
AS14.29         41   0.5646   0.5038     0.892
AS14.27         41   0.4465   0.3641     0.815
AS14.28         38   0.5306   0.4319     0.814
AS14.16         45   0.4487   0.3572     0.796
AS14.26         68   0.5308   0.4134     0.779
AS14.24         49   0.4237   0.3255     0.768
AS14.14         44   0.4339   0.3215     0.741
AS14.23         43   0.3513   0.2542     0.724
AS14.25         27   0.5380   0.3793     0.705
AS14.03         46   0.4920   0.3439     0.699
AS14.08         52   0.4

#### 7. Summary & Implications

In [7]:
print(f"\n{'='*65}")
print("Task 5B — Summary of Findings")
print(f"{'='*65}")
print(f"""
1. TOP-5% ERROR DOMINANCE
   For both XGBoost and LSTM, the worst 5% of predictions account for
   ~19% of MAE but ~41% of MSE — an amplification ratio of ~2.2×.
   This ratio is remarkably stable across both models AND the baselines
   from EDA (b_09), confirming it is a structural property of the mood
   prediction task, not an artifact of any specific model.

2. PRACTICAL INTERPRETATION
   Under MAE, model quality is judged by typical prediction accuracy.
   Under MSE, model quality is disproportionately determined by the
   ~60 worst predictions (top 5% of ~1,200 test instances). A clinician
   using these mood predictions would likely prefer MAE as the evaluation
   metric, because a model that is slightly worse on average but avoids
   catastrophic mispredictions is clinically preferable.

3. MODEL RANKING STABILITY
   Both MAE and MSE rank {mae_winner} as the better model on pooled
   metrics. The metrics agree in this case because the error distributions
   of both models have similar shapes (similar top-5% shares). Ranking
   disagreement would occur if one model had lower typical errors but
   higher tail errors than the other.

4. PER-PATIENT HETEROGENEITY
   The MSE/MAE ratio varies substantially across patients (range shown
   above). Patients with high ratios have occasional large errors that
   MSE amplifies. This matters for clinical deployment: a model that
   appears acceptable under global MSE may perform poorly for specific
   patients whose error distribution is heavy-tailed.

5. CONNECTION TO TASK 5A
   The example from 5A (identical |errors| → identical rankings) is
   the theoretical limiting case. In practice, mood prediction errors
   have high variance (std ≈ 0.46–0.48), so the metrics DO behave
   differently. The wider the error distribution, the more MSE diverges
   from MAE — and the more the choice of metric matters.
""")

# Save summary table
summary = pd.DataFrame({
    "metric": ["MAE", "MSE", "Top 5% share of MAE", "Top 5% share of MSE",
               "MSE/MAE sensitivity ratio"],
    "XGBoost": [
        results["XGBoost"]["MAE"],
        results["XGBoost"]["MSE"],
        f"{results['XGBoost']['abs_err'][results['XGBoost']['abs_err'] >= np.percentile(results['XGBoost']['abs_err'], 95)].sum() / results['XGBoost']['abs_err'].sum() * 100:.1f}%",
        f"{results['XGBoost']['sq_err'][results['XGBoost']['abs_err'] >= np.percentile(results['XGBoost']['abs_err'], 95)].sum() / results['XGBoost']['sq_err'].sum() * 100:.1f}%",
        f"{(results['XGBoost']['sq_err'][results['XGBoost']['abs_err'] >= np.percentile(results['XGBoost']['abs_err'], 95)].sum() / results['XGBoost']['sq_err'].sum()) / (results['XGBoost']['abs_err'][results['XGBoost']['abs_err'] >= np.percentile(results['XGBoost']['abs_err'], 95)].sum() / results['XGBoost']['abs_err'].sum()):.2f}×",
    ],
    "LSTM": [
        results["LSTM"]["MAE"],
        results["LSTM"]["MSE"],
        f"{results['LSTM']['abs_err'][results['LSTM']['abs_err'] >= np.percentile(results['LSTM']['abs_err'], 95)].sum() / results['LSTM']['abs_err'].sum() * 100:.1f}%",
        f"{results['LSTM']['sq_err'][results['LSTM']['abs_err'] >= np.percentile(results['LSTM']['abs_err'], 95)].sum() / results['LSTM']['sq_err'].sum() * 100:.1f}%",
        f"{(results['LSTM']['sq_err'][results['LSTM']['abs_err'] >= np.percentile(results['LSTM']['abs_err'], 95)].sum() / results['LSTM']['sq_err'].sum()) / (results['LSTM']['abs_err'][results['LSTM']['abs_err'] >= np.percentile(results['LSTM']['abs_err'], 95)].sum() / results['LSTM']['abs_err'].sum()):.2f}×",
    ],
})
summary.to_csv(OUTPUT_DIR / "metric_sensitivity_summary.csv", index=False)
print(f"Saved metric_sensitivity_summary.csv to {OUTPUT_DIR}")
print(f"\n✅ Task 5 complete.")


Task 5B — Summary of Findings

1. TOP-5% ERROR DOMINANCE
   For both XGBoost and LSTM, the worst 5% of predictions account for
   ~19% of MAE but ~41% of MSE — an amplification ratio of ~2.2×.
   This ratio is remarkably stable across both models AND the baselines
   from EDA (b_09), confirming it is a structural property of the mood
   prediction task, not an artifact of any specific model.

2. PRACTICAL INTERPRETATION
   Under MAE, model quality is judged by typical prediction accuracy.
   Under MSE, model quality is disproportionately determined by the
   ~60 worst predictions (top 5% of ~1,200 test instances). A clinician
   using these mood predictions would likely prefer MAE as the evaluation
   metric, because a model that is slightly worse on average but avoids
   catastrophic mispredictions is clinically preferable.

3. MODEL RANKING STABILITY
   Both MAE and MSE rank LSTM as the better model on pooled
   metrics. The metrics agree in this case because the error distributions